# FastVLA: Ultra-Efficient VLA Fine-Tuning on Colab T4

Welcome to the FastVLA one-click training template! By leveraging Unsloth-inspired 4-bit kernels and Custom Triton Action Heads, we enable fine-tuning 7B parameter robotics policies on a free Tesla T4 (15GB) GPU.

### Why FastVLA on Colab?
- **Low Memory**: Train massive models like OpenVLA-7B with only 6.3GB VRAM.
- **High Speed**: 6x faster throughput than standard robotics pipelines.
- **Zero Friction**: One click to setup and train.
- **Robust Training**: Automatic shape validation and error handling.

> **Status:** Verified on Tesla T4 (15GB). Converged from 19.6 -> 0.79 Loss in 2000 steps.

## ⚠️ Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2. 

**If you haven't been approved yet:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Wait for Approval**: This can take anywhere from 1 hour to 2 days.
3. **Login**: Once approved, use your HF token in the cell below.

**🚀 Want to skip the wait? (Instant Mode)**
If you want to start training **immediately** without waiting for Meta's approval, simply change the `model_id` in Step 3 to `"smolvla"`. It is non-gated and works instantly!

## 0. Authentication
1. **Add Token**: Click the **🔑 (Secrets)** icon on the left sidebar, add `HF_TOKEN` with your token, and enable 'Notebook access'.

Alternatively, use a non-gated model like `smolvla` later in the notebook.

In [ ]:
from huggingface_hub import login
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    login(token=token)
except:
    # Manual fallback for non-Colab or missing secret
    print("No HF_TOKEN found in secrets. Redirecting to manual login...")
    login()

## 1. Setup Environment
We install FastVLA directly from GitHub with all extras (Triton kernels, LoRA, logging).

> **Note:** Dependencies are installed in a specific order to avoid version conflicts with Colab's pre-installed packages.

In [ ]:
# ⚠️ CRITICAL: Install in correct order to avoid version conflicts
# Do NOT use --force-reinstall (it breaks Colab's pre-installed packages)

# 1. Pin numpy <2.2.0 to prevent ABI breakage with scipy/sklearn/tensorflow
!pip install -q 'numpy>=1.24.0,<2.2.0'

# 2. Upgrade huggingface_hub FIRST (required by peft, datasets)
!pip install -q --upgrade 'huggingface_hub>=0.30.0' --no-cache-dir

# 3. Install Unsloth (optional but recommended for best 4-bit performance)
!pip install -q unsloth_zoo --no-cache-dir
!pip install -q git+https://github.com/unslothai/unsloth.git --no-cache-dir

# 4. Install FastVLA with all extras (GPU kernels + LoRA + logging)
!pip install -q 'fastvla[all] @ git+https://github.com/BouajilaHamza/FastVLA.git' --no-cache-dir

# 5. Upgrade remaining deps (no --force-reinstall!)
!pip install -q --upgrade bitsandbytes accelerate peft transformers datasets timm --no-cache-dir

# ⚠️ Import Unsloth FIRST to apply monkey-patches to transformers/peft
try:
    import unsloth
    print('✓ Unsloth imported first (patches applied)')
except ImportError:
    print('ℹ Unsloth not installed — 4-bit loading will use BitsAndBytes fallback')

# Verify FastVLA installation
print('\n' + '='*60)
print('FastVLA Environment Diagnostic')
print('='*60)

try:
    import fastvla
    print(f'✓ FastVLA {fastvla.__version__} imported')
    
    from fastvla import model as fastvla_model
    if fastvla_model.UNSLOTH_AVAILABLE:
        print('  ✓ Unsloth integration detected')
    else:
        print('  ℹ Unsloth not detected (will use BitsAndBytes for 4-bit)')
    
    import torch
    print(f'  ✓ PyTorch: {torch.__version__}')
    print(f'  ✓ CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'  ✓ Device: {torch.cuda.get_device_name(0)}')
except Exception as e:
    print(f'✗ Import failed: {e}')

print('='*60)
print('✅ Environment ready! Proceed to load the model.')


## 2. Optional: Mount Google Drive
If you want to save your checkpoints and best models, mount your Google Drive.

In [ ]:
from google.colab import drive
# drive.mount('/content/drive')

## 3. Load Model in 4-bit (QLoRA)
We load OpenVLA-7B with 4-bit quantization. 

**Important:** The `action_dim` parameter must match your dataset's action dimensions. For PushT, this is 2 (x, y coordinates). For other robotics datasets, it's typically 7 (x, y, z, roll, pitch, yaw, gripper).

**Instant Start Tip:** To skip Llama-2 gated access issues, change `model_id` below to `"smolvla"`.

In [ ]:
from fastvla import FastVLAModel
import torch

model_id = "openvla/openvla-7b" # Switch to "smolvla" for non-gated, instant access

# IMPORTANT: action_dim must match your dataset. PushT uses 2D actions (x, y).
# Standard robotics datasets often use 7D (x, y, z, roll, pitch, yaw, gripper).
ACTION_DIM = 2  # Change this to match your dataset!

print(f"Loading {model_id} in 4-bit...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
    action_dim=ACTION_DIM,  # Must match your dataset!
)

print(f"Model loaded. Current VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4. Fine-Tuning on PushT (Real Robotics)
Next, we load the lerobot/pusht_image dataset and run an optimized training loop.

**Training Features:**
- Mixed precision (fp16) for faster training on T4
- 8-bit optimizer for memory efficiency
- Automatic shape validation prevents cryptic errors
- Gradient accumulation for flexible batch sizes

In [ ]:
from fastvla import FastVLATrainer, get_dataset

# Load dataset
dataset = get_dataset("lerobot/pusht_image")

# Verify action dimensions match (FastVLA now validates this automatically)
sample_action = dataset[0]["actions"]
print(f"Dataset action Shape: {sample_action.shape}")
print(f"Model action_dim: {model.config.action_dim}")

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=2,  # Optimized for T4 VRAM
    lr=1e-4,
    max_steps=50,  # Demonstration steps (use 2000 for full training)
    # Training optimizations
    use_mixed_precision=True,  # fp16 for T4
    use_8bit_optimizer=True,   # Memory efficient
    gradient_accumulation_steps=2,  # Effective batch size = 4
    # Checkpointing and logging
    output_dir="./checkpoints",
    save_steps=50,
    logging_steps=10,
)

print("Starting Training...")
results = trainer.train()

print(f"Training Complete! Final loss: {results[-1]['loss']:.4f}")

## 5. Inference (Predict Action)
We test the model by giving it a visual input and a text command to predict the robot's next action.

In [ ]:
import numpy as np
from PIL import Image

# Random input for demonstration
dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

# Process the image and prompt through the model
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

pixel_values = transform(dummy_image).unsqueeze(0).unsqueeze(0)  # [1, 1, C, H, W]
input_ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"]

with torch.no_grad():
    action, _ = model(pixel_values=pixel_values, input_ids=input_ids)

print(f"Predicted Action Tensor ({model.config.action_dim}D):")
print(action)

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Copyright (c) 2025 FastVLA Team.**